# S02 — Shoulder: raw vs curve vs I_ML, front and side views separately (PC)

Inputs: `data/fit3d/feat_rehab24_shoulder.npz` (S01b, Colab) and `data/fit3d/feat_fit3d_shoulder.npz` (S01).
Both were built frame by frame with the live feature code, so every result here is a **live** result.

| method | what it is |
|---|---|
| `raw` | the angle the live system uses today (hip–shoulder–elbow in the image) |
| `elev2d` | trunk axis vs upper arm in the image (no learning) |
| `curve` | monotonic curve on `raw`, one per camera view |
| `iml` | LightGBM on the 15 shoulder features, **one model for every view** |
| `iml_view` | one LightGBM per view (reported, to answer "one model or one per view?") |

Tests are always on people the model never saw: **E1** Fit3D leave-one-person-out, **E2a** REHAB → Fit3D,
**E2b** Fit3D → REHAB, **E3** both datasets leave-one-person-out. Fit3D has no side camera, so the
side (profile) view is tested on REHAB24-6 only.

**Decision rule, fixed before the results** (applied to the front and the profile view separately):

* use `iml` if, in E3, it beats the best of `raw`/`elev2d`/`curve` by **≥ 3°** (per-person median gain),
  with Wilcoxon **p < 0.05**, and is also better in every cross-dataset test that has that view;
* otherwise use whichever of `elev2d` / `curve` beats `raw` by more (≥ 2° per person in E3); otherwise keep `raw`;
* `iml_view` replaces `iml` only if it is ≥ 1° better than `iml` on both views.

The deployable model and curves are trained on **REHAB24-6 only** (Fit3D licence).

In [1]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
import lightgbm as lgb

REPO  = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'   # <-- your repo
FILES = {'rehab24': os.path.join(REPO, 'data', 'fit3d', 'feat_rehab24_shoulder.npz'),
         'fit3d':   os.path.join(REPO, 'data', 'fit3d', 'feat_fit3d_shoulder.npz')}
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
sys.path[:0] = [REPO, os.path.join(REPO, 'scripts')]
from calibration import fit_curve, apply_calibration
from fit_calibration import extend_below
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)

def view_class(v):
    '''camera-relative view (REHAB24-6 cam18 labels describe the subject's pose to cam17)'''
    v = str(v).lower(); cam18 = 'cam18' in v
    if 'half' in v: return 'half'
    if 'front' in v: return 'profile' if cam18 else 'front'
    if 'prof' in v or 'side' in v: return 'front' if cam18 else 'profile'
    return 'unknown'

frames = []
for name, path in FILES.items():
    if not os.path.exists(path):
        print('missing (skipped):', path); continue
    d = np.load(path, allow_pickle=False)
    names = [str(n) for n in d['feature_names']]
    df = pd.DataFrame(d['X'], columns=names)
    for k in ('y', 'person', 'recording', 'view', 'exercise', 'dataset'):
        df[k] = d[k]
    frames.append(df)
D = pd.concat(frames, ignore_index=True)
D['vclass'] = D['view'].map(view_class)
D = D[np.isfinite(D['angle_2d']) & np.isfinite(D['y'])].reset_index(drop=True)
FEATS = names
print(D.groupby(['dataset', 'vclass']).agg(frames=('y', 'size'), people=('person', 'nunique'),
      recordings=('recording', 'nunique'), truth_p5=('y', lambda s: s.quantile(.05)),
      truth_p95=('y', lambda s: s.quantile(.95))).round(1))
# check of the camera-relative labels: torso_ratio is large facing the camera, small side-on
print(D.groupby(['dataset', 'view', 'vclass'])['torso_ratio'].median().round(2))

                 frames  people  recordings  truth_p5  truth_p95
dataset vclass                                                  
fit3d   front    133916       8         311      19.8      142.2
        half       1757       3           5      22.4      135.6
rehab24 front     93725      18          28      14.2      139.7
        half      35289       7          17      15.8      141.8
        profile   33292       8           8      23.2      143.8
dataset  view                vclass 
fit3d    front               front      0.60
         half                half       0.61
rehab24  Front               front      0.59
         Front/cam18         profile    0.16
         Half-profile        half       0.41
         Half-profile/cam18  half       0.54
Name: torso_ratio, dtype: float32


## Methods

In [2]:
HP = dict(objective='l1', n_estimators=200, num_leaves=31, learning_rate=0.09, min_child_samples=80,
          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbosity=-1, n_jobs=N_JOBS)

def fit_lgb(tr):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return lgb.LGBMRegressor(**HP).fit(tr[FEATS].to_numpy(), tr['y'])

def one_curve(raw, truth):
    for mc in (300, 100, 30):
        try:
            return extend_below(fit_curve(raw, truth, min_count=mc))
        except ValueError:
            pass

def curve_pred(tr, te):
    pooled = one_curve(tr['angle_2d'], tr['y']); out = np.empty(len(te))
    for vc, idx in te.groupby('vclass').indices.items():
        sub = tr[tr['vclass'] == vc]
        p = one_curve(sub['angle_2d'], sub['y']) if len(sub) >= 2000 else None
        out[idx] = apply_calibration(te['angle_2d'].to_numpy(float)[idx], p or pooled)
    return out

def run(train_mask, test_mask, label):
    tr, te = D[train_mask], D[test_mask].reset_index(drop=True)
    out = te[['dataset', 'person', 'recording', 'vclass', 'exercise', 'y']].copy()
    out['raw'] = te['angle_2d'].to_numpy(float)
    out['elev2d'] = te['elev_2d'].to_numpy(float)
    out['curve'] = curve_pred(tr, te)
    pooled = fit_lgb(tr)
    out['iml'] = np.clip(pooled.predict(te[FEATS].to_numpy()), 0, 180)
    out['iml_view'] = out['iml'].to_numpy().copy()
    for vc, idx in te.groupby('vclass').indices.items():
        sub = tr[tr['vclass'] == vc]
        if len(sub) >= 2000:
            out.loc[idx, 'iml_view'] = np.clip(fit_lgb(sub).predict(te.iloc[idx][FEATS].to_numpy()), 0, 180)
    out['experiment'] = label
    return out

t0 = time.time()
is_fit, is_reh = D['dataset'] == 'fit3d', D['dataset'] == 'rehab24'
parts = []
if is_fit.any():
    for p in sorted(D.loc[is_fit, 'person'].unique()):
        parts.append(run(is_fit & (D['person'] != p), D['person'] == p, 'E1 Fit3D LOSO'))
if is_fit.any() and is_reh.any():
    parts += [run(is_reh, is_fit, 'E2a REHAB->Fit3D'), run(is_fit, is_reh, 'E2b Fit3D->REHAB')]
for p in sorted(D['person'].unique()):
    parts.append(run(D['person'] != p, D['person'] == p, 'E3 both LOSO'))
    print('  held out', p, f'{time.time() - t0:.0f}s', flush=True)
P = pd.concat(parts, ignore_index=True)
print(f'finished in {(time.time() - t0) / 60:.1f} min')

  held out fit3d_s03 113s
  held out fit3d_s04 127s
  held out fit3d_s05 172s
  held out fit3d_s07 207s
  held out fit3d_s08 220s
  held out fit3d_s09 230s
  held out fit3d_s10 240s
  held out fit3d_s11 249s
  held out rehab_PM_000 260s
  held out rehab_PM_001 269s
  held out rehab_PM_002 275s
  held out rehab_PM_003 286s
  held out rehab_PM_004 295s
  held out rehab_PM_012 320s
  held out rehab_PM_013 339s
  held out rehab_PM_014 368s
  held out rehab_PM_016 380s
  held out rehab_PM_023 393s
  held out rehab_PM_024 402s
  held out rehab_PM_025 417s
  held out rehab_PM_026 427s
  held out rehab_PM_032 442s
  held out rehab_PM_033 456s
  held out rehab_PM_039 468s
  held out rehab_PM_040 492s
  held out rehab_PM_101 504s
  held out rehab_PM_102 529s
  held out rehab_PM_109 575s
  held out rehab_PM_110 613s
  held out rehab_PM_114 646s
  held out rehab_PM_115 664s
  held out rehab_PM_122 679s
  held out rehab_PM_123 687s
finished in 11.5 min


## Results

In [3]:
M = ['raw', 'elev2d', 'curve', 'iml', 'iml_view']
err = P[M].sub(P['y'], axis=0).abs()
rec = pd.concat([P[['experiment', 'dataset', 'person', 'recording', 'vclass', 'exercise']], err], axis=1) \
        .groupby(['experiment', 'dataset', 'person', 'recording', 'vclass', 'exercise'], as_index=False)[M].mean()
tab = lambda r: r.groupby('experiment')[M].median().round(2).assign(recordings=r.groupby('experiment').size())
for vc in ('front', 'half', 'profile'):
    r = rec[rec.vclass == vc]
    if len(r):
        print(f'\n{vc.upper()} VIEW - median per-recording MAE (deg)'); display(tab(r))
print('\nBY EXERCISE, E3'); display(rec[rec.experiment == 'E3 both LOSO'].groupby(['exercise', 'vclass'])[M].median().round(2))
P['band'] = pd.cut(P['y'], [0, 30, 60, 90, 120, 150, 181], labels=['<30', '30-60', '60-90', '90-120', '120-150', '>150'])
print('\nBY TRUE ANGLE, E3 (mean abs error)')
display(pd.concat([P[['experiment', 'band']], err], axis=1)[P.experiment == 'E3 both LOSO']
        .groupby('band', observed=True)[M].mean().round(2))


FRONT VIEW - median per-recording MAE (deg)


,raw,elev2d,curve,iml,iml_view,recordings
experiment,,,,,,
E1 Fit3D LOSO,15.13,14.39,7.17,5.11,5.24,311
E2a REHAB->Fit3D,15.13,14.39,9.34,7.73,7.53,311
E2b Fit3D->REHAB,10.82,10.38,9.00,8.11,8.38,28
E3 both LOSO,14.43,14.00,7.85,5.56,5.36,339



HALF VIEW - median per-recording MAE (deg)


,raw,elev2d,curve,iml,iml_view,recordings
experiment,,,,,,
E1 Fit3D LOSO,20.76,20.65,10.87,7.50,7.50,5
E2a REHAB->Fit3D,20.76,20.65,14.81,8.04,8.05,5
E2b Fit3D->REHAB,17.40,17.56,12.73,9.97,9.97,17
E3 both LOSO,18.15,18.24,9.51,6.92,5.97,22



PROFILE VIEW - median per-recording MAE (deg)


,raw,elev2d,curve,iml,iml_view,recordings
experiment,,,,,,
E2b Fit3D->REHAB,27.89,27.27,16.39,13.45,13.45,8
E3 both LOSO,27.89,27.27,16.36,9.11,8.58,8



BY EXERCISE, E3


raw  elev2d  curve   iml  iml_view
exercise                         vclass                                       
dumbbell_overhead_shoulder_press front    17.10   15.26   9.10  6.11      5.36
dumbbell_scaptions               front    12.02   12.59   5.61  4.60      4.62
ex1                              front     7.50    7.99   5.73  3.69      3.66
                                 half     11.64   13.30   7.58  6.58      4.07
ex2                              front    12.12   11.11   7.15  4.38      4.86
                                 half     18.15   18.24   9.09  6.92      7.25
                                 profile  27.89   27.27  16.36  9.11      8.58
neutral_overhead_shoulder_press  front    17.86   17.35  10.07  6.51      5.86
overhead_trap_raises             front    23.22   23.44  12.01  6.77      6.37
                                 half     32.87   31.51  16.23  9.04      7.78
side_lateral_raise               front     8.49   10.48   4.76  4.44      4.67
                                 half     16.64   17.28  11.95  8.22     10.15
w_raise                          front     7.62   10.46   5.65  5.46      5.28


BY TRUE ANGLE, E3 (mean abs error)


,raw,elev2d,curve,iml,iml_view
band,,,,,
<30,4.70,9.37,4.59,3.96,4.09
30-60,10.74,13.25,8.68,6.26,6.42
60-90,17.38,16.54,12.54,6.55,6.47
90-120,23.89,20.19,11.14,6.82,6.55
120-150,23.91,20.64,9.15,6.09,7.08
>150,15.89,14.29,18.97,15.23,17.55


## Decision (rule fixed in advance)

In [4]:
def per_person(r, a, b):
    s = r.groupby('person')[[a, b]].mean()
    g = s[a] - s[b]
    p = wilcoxon(g).pvalue if len(s) >= 6 and (g != 0).any() else float('nan')
    return float(g.median()), int((g > 0).sum()), len(s), float(p)

CHOICE, REPORT = {}, []
e3 = rec[rec.experiment == 'E3 both LOSO']
for vc in ('front', 'profile'):
    r = e3[e3.vclass == vc]
    if r.empty:
        print(f'{vc}: no recordings'); continue
    base = min(('raw', 'elev2d', 'curve'), key=lambda m: r[m].median())
    gain, better, n_p, p = per_person(r, base, 'iml')
    cross = [x for x in ('E2a REHAB->Fit3D', 'E2b Fit3D->REHAB')
             if len(rec[(rec.experiment == x) & (rec.vclass == vc)])]
    cross_ok = all(rec[(rec.experiment == x) & (rec.vclass == vc)]['iml'].median()
                   < rec[(rec.experiment == x) & (rec.vclass == vc)][base].median() for x in cross)
    if gain >= 3.0 and p < 0.05 and cross_ok:
        CHOICE[vc] = 'iml'
    else:
        gains = {m: per_person(r, 'raw', m)[0] for m in ('elev2d', 'curve')}
        best = max(gains, key=gains.get)
        CHOICE[vc] = best if gains[best] >= 2.0 else 'raw'
    REPORT.append(dict(view=vc, people=n_p, baseline=base, baseline_mae=round(r[base].median(), 2),
                       iml_mae=round(r['iml'].median(), 2), gain_per_person=round(gain, 2),
                       iml_better_for=f'{better}/{n_p}', wilcoxon_p=round(p, 4),
                       cross_dataset_tests=', '.join(cross) or 'none', cross_ok=cross_ok, choice=CHOICE[vc]))
display(pd.DataFrame(REPORT))

per_view_better = all(len(e3[e3.vclass == vc]) and
                      e3[e3.vclass == vc]['iml'].median() - e3[e3.vclass == vc]['iml_view'].median() >= 1.0
                      for vc in ('front', 'profile'))
if per_view_better and 'iml' in CHOICE.values():
    CHOICE = {vc: ('iml_view' if c == 'iml' else c) for vc, c in CHOICE.items()}
print('one model or one per view?', 'one per view' if per_view_better else 'one model for every view')
print('CHOICE:', CHOICE)

,view,people,baseline,baseline_mae,iml_mae,gain_per_person,iml_better_for,wilcoxon_p,cross_dataset_tests,cross_ok,choice
0,front,26,curve,7.85,5.56,2.55,25/26,0.0000,"E2a REHAB->Fit3D, E2b Fit3D->REHAB",True,curve
1,profile,8,curve,16.36,9.11,6.08,8/8,0.0078,E2b Fit3D->REHAB,True,iml


one model or one per view? one model for every view
CHOICE: {'front': 'curve', 'profile': 'iml'}


## Save what was chosen (trained on REHAB24-6 only)

In [5]:
reh = D[D['dataset'] == 'rehab24']
saved = []
if any(c.startswith('iml') for c in CHOICE.values()):
    path = os.path.join(REPO, 'data', 'iml_shoulder_rehab24.txt')
    fit_lgb(reh).booster_.save_model(path)
    json.dump({'joint': 'shoulder', 'features': FEATS, 'feature_code': 'shoulder_features.ShoulderFeatureStream',
               'params': {k: v for k, v in HP.items() if k != 'n_jobs'}, 'choice_per_view': CHOICE,
               'trained_on': 'REHAB24-6 Ex1 + Ex2 only (CC BY-NC), streaming features',
               'cv_E3_median_mae': {r['view']: r['iml_mae'] for r in REPORT}},
              open(path + '.json', 'w'), indent=1)
    saved.append(path)
    if 'iml_view' in CHOICE.values():
        for vc in ('front', 'profile'):
            sub = reh[reh.vclass == vc]
            if len(sub) >= 2000:
                pv = os.path.join(REPO, 'data', f'iml_shoulder_rehab24_{vc}.txt')
                fit_lgb(sub).booster_.save_model(pv); saved.append(pv)
for vc, c in CHOICE.items():
    if c == 'curve':
        sub = reh[reh.vclass == vc]
        params = one_curve(sub['angle_2d'], sub['y'])
        if params:
            pc = os.path.join(REPO, 'data', f'population_curve_shoulder_{vc}.json')
            json.dump({**params, 'joint': 'shoulder', 'view': vc, 'trained_on': 'REHAB24-6 Ex1 + Ex2'},
                      open(pc, 'w'), indent=1)
            saved.append(pc)
print('saved:', *saved, sep='\n  ') if saved else print('nothing to save')
if 'elev2d' in CHOICE.values():
    print('elev2d chosen for', [v for v, c in CHOICE.items() if c == 'elev2d'],
          '- no model needed: the live angle becomes the trunk-axis angle (a small demo_cv change)')
rec.to_csv(os.path.join(REPO, 'data', 'fit3d', 'shoulder_results_per_recording.csv'), index=False)

saved:
  F:\namal sir project\github repo\AI-based-Physiotherapy-project-\data\iml_shoulder_rehab24.txt
  F:\namal sir project\github repo\AI-based-Physiotherapy-project-\data\population_curve_shoulder_front.json
